# Hand Gesture Recognition - Local Version
로컬 환경에서 실행되는 버전입니다. Google Colab 의존성이 제거되고, 웹캠 녹화가 OpenCV 직접 캡처로 대체되었습니다.

**사전 요구사항:**
- Step 1에서 필요한 Python 패키지를 먼저 설치하세요.
- `ffmpeg`는 **Step 6 (긴 영상 분석)에서만** 필요합니다. 기본 기능(Step 1~5)은 ffmpeg 없이 동작합니다.

# Step 1 - 패키지 설치

In [ ]:
!pip install pandas opencv-python numpy torch mediapipe==0.10.14 tabulate alive-progress

# Step 2 - 초기화

#### Import Libraries

In [34]:
import pandas as pd
import os
import cv2 as cv
import numpy as np
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import pickle
from torch.autograd import Variable
from tabulate import tabulate
from IPython.display import clear_output
from alive_progress import alive_bar
import mediapipe as mp

#### Assign device

In [35]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


#### Initialize functions

In [36]:
def initialize_model(n_way, k_shot, classes):
  """
  main function to initialize both model components based on a given n_way, k_shot scenario
  Args:
    n_way (int): Number of different classes to be classfied
    k_shot (int): Number of support samples per class
    classes (list): Names of classes to predict
  Returns:
    feature_encoder (LSTMEncoder-object): initialized LSTMEncoder-object
    relation_network (RelationNetwork-object): initialized RelationNetwork-object
  """

  # Check if classes-length match n_way
  if n_way != len(classes):
    print('Error: The size of your list of gestures does not match N_WAY. Please correct this and restart the process.')
    return None, None

  # Clone github repo where the model is stored
  succes = os.system("""git clone https://github.com/nielsschluesener/S-STRHanGe.git""")

  # define model_name and its path
  model_name = f'{n_way}way-{k_shot}shot'
  model_path = 'S-STRHanGe/deployment/models/'

  # get deployment parameters
  if os.path.isfile(os.path.join(model_path, f'{model_name}_deployment_param.pkl')):
    with open(os.path.join(model_path, f'{model_name}_deployment_param.pkl'), 'rb') as f:
      deployment_param = pickle.load(f)

  # create feature_encoder and relation_network objects with params given by deployment_params
  feature_encoder = LSTMEncoder(deployment_param['feature_length'], deployment_param['num_units_lstm_encoder'], deployment_param['num_lstm_layer_encoder']).to(device)
  relation_network = RelationNetwork(deployment_param['num_units_lstm_encoder']*2, deployment_param['num_units_lstm_relationnet'], deployment_param['num_units_fc_relu'], deployment_param['num_lstm_layer_relationnet']).to(device)

  # load the trained weights
  feature_encoder.load_state_dict(torch.load(os.path.join(str(os.path.join(model_path, f'{model_name}_feature_encoder.pkl'))), map_location=device))
  relation_network.load_state_dict(torch.load(os.path.join(str(os.path.join(model_path, f'{model_name}_relation_network.pkl'))), map_location=device))

  return feature_encoder, relation_network


class LSTMEncoder(nn.Module):
  """First part of the model, which encodes the sequence of hand gesture keypoints"""
  def __init__(self, input_size, hidden_lstm_size, num_lstm_layer):
    super(LSTMEncoder, self).__init__()
    self.input_size = input_size
    self.hidden_lstm_size = hidden_lstm_size
    self.num_lstm_layer = num_lstm_layer
    self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_lstm_size, num_layers=num_lstm_layer, batch_first=True)

  def forward(self, x):
    h0 = torch.zeros(self.num_lstm_layer, x.size(0), self.hidden_lstm_size).to(device)
    c0 = torch.zeros(self.num_lstm_layer, x.size(0), self.hidden_lstm_size).to(device)
    out, _ = self.lstm(x, (h0, c0))
    return out


class RelationNetwork(nn.Module):
  """Second part of the model, which calculates the relation-scores"""
  def __init__(self, input_size, hidden_lstm_size, fc_sizes: list, num_lstm_layer):
    super(RelationNetwork, self).__init__()
    self.input_size = input_size
    self.hidden_lstm_size = hidden_lstm_size
    self.layers = nn.ModuleList()
    self.num_lstm_layer = num_lstm_layer
    self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_lstm_size, num_layers=num_lstm_layer, batch_first=True)
    input_size = hidden_lstm_size
    for size in fc_sizes:
      self.layers.append(nn.Linear(input_size, size))
      self.layers.append(nn.ReLU())
      input_size = size
    self.end_layer = nn.Sequential(nn.Linear(input_size, 1), nn.Sigmoid())

  def forward(self, x):
    h0 = torch.zeros(self.num_lstm_layer, x.size(0), self.hidden_lstm_size).to(device)
    c0 = torch.zeros(self.num_lstm_layer, x.size(0), self.hidden_lstm_size).to(device)
    out, _ = self.lstm(x, (h0, c0))
    out = out[:, -1, :]
    for layer in self.layers:
      out = layer(out)
    out = self.end_layer(out)
    return out

In [37]:
def record_frames(frames_dir, num_frames=72):
  """
  웹캠에서 직접 프레임을 JPEG로 캡처하여 저장합니다.
  VideoWriter 코덱 문제를 우회하기 위해 비디오 파일을 거치지 않습니다.
  Args:
    frames_dir (str): JPEG 프레임을 저장할 디렉토리
    num_frames (int): 저장할 총 프레임 수 (기본값: 72 = 3초@24fps)
  Returns:
    True on success, False on failure
  """
  os.makedirs(frames_dir, exist_ok=True)

  cap = cv.VideoCapture(0)
  if not cap.isOpened():
    print('Error: Cannot open webcam.')
    return False

  # 3초 카운트다운
  for i in range(3, 0, -1):
    deadline = time.time() + 1.0
    while time.time() < deadline:
      ret, frame = cap.read()
      if not ret:
        break
      frame = cv.flip(frame, 1)
      cv.putText(frame, f'Recording starts in {i}', (30, 60),
                 cv.FONT_HERSHEY_SIMPLEX, 1.4, (0, 255, 0), 3)
      cv.imshow('Hand Gesture Recorder  [Q to cancel]', frame)
      if cv.waitKey(30) & 0xFF == ord('q'):
        cap.release()
        cv.destroyAllWindows()
        return False

  # num_frames 프레임을 24fps 기준으로 캡처
  frame_interval = 1.0 / 24.0
  saved = 0
  next_capture = time.time()

  while saved < num_frames:
    ret, frame = cap.read()
    if not ret:
      break
    frame = cv.flip(frame, 1)
    current_time = time.time()

    if current_time >= next_capture:
      # 훈련 데이터와 동일한 해상도(360x240)로 리사이즈 후 저장
      resized = cv.resize(frame, (360, 240))
      cv.imwrite(os.path.join(frames_dir, f'frame_{str(saved + 1).zfill(4)}.jpg'), resized)
      saved += 1
      next_capture += frame_interval

    remaining = max(0, (num_frames - saved) * frame_interval)
    cv.putText(frame, f'Recording ends in {remaining:.1f}s', (30, 60),
               cv.FONT_HERSHEY_SIMPLEX, 1.4, (0, 0, 255), 3)
    cv.imshow('Hand Gesture Recorder  [Q to cancel]', frame)
    if cv.waitKey(1) & 0xFF == ord('q'):
      break

  cap.release()
  cv.destroyAllWindows()
  print(f'  Captured {saved}/{num_frames} frames.')
  return saved > 0


def extract_keypoints(results):
  """
  Extracts keypoints from a given mediapipe-Hands-result
  """
  if results.multi_hand_landmarks:
    landmarks = np.array([[res.x, res.y, res.z] for res in results.multi_hand_landmarks[0].landmark]).flatten()
    return landmarks
  else:
    return np.zeros(63)


def move_values(data, seq_length, direction='back'):
  """
  Move values in each sequence either towards the back or front.
  """
  max_values = data.max(axis=1)
  if np.array_equal(max_values, np.zeros(seq_length)):
    return np.zeros((seq_length, 63))
  else:
    i_first_value = next(i for i, v in enumerate(max_values) if v != 0)
    i_last_value  = next(i for i, v in reversed(list(enumerate(max_values))) if v != 0)
    non_zeros = data[i_first_value:i_last_value + 1]
    zeros = np.zeros((seq_length - non_zeros.shape[0], 63))
    if direction == 'back':
      return np.concatenate((zeros, non_zeros), axis=0)
    elif direction == 'front':
      return np.concatenate((non_zeros, zeros), axis=0)


def initialize_supset_dir(n_way, k_shot, supset_dir):
  """
  Initializes the support_set directory
  """
  if os.path.exists(supset_dir):
    safety_check = input('Caution: This step will delete the old support set. Do you want to continue? Type Yes or No and hit Enter.')
    if safety_check == 'Yes':
      os.system('rm -rf support_set' if os.name != 'nt' else 'rmdir /s /q support_set')
      clear_output()
    elif safety_check == 'No':
      support_x, support_y = create_supset_from_keypoints(n_way, k_shot, supset_dir)
      print('Procedure canceled. Old support set was preserved.')
      return support_x, support_y

  if not os.path.exists(supset_dir):
    os.makedirs(supset_dir)
    for n in range(1, n_way + 1):
      for s in range(1, k_shot + 1):
        os.makedirs(supset_dir + f'/class{n}_sample{s}/frames')
        os.makedirs(supset_dir + f'/class{n}_sample{s}/keypoints')

  return None, None


def generate_supset_keypoints_from_frame(n_way, k_shot, supset_dir):
  """
  Generates and saves keypoints from given supset-frames
  """
  num_frames = 72
  detected = 0
  with mp.solutions.hands.Hands(
      static_image_mode=True,
      max_num_hands=1,
      min_detection_confidence=0.5) as hands:
    for frame_num in range(1, num_frames + 1):
      image_path = supset_dir + f'/class{n_way}_sample{k_shot}/frames/frame_{str(frame_num).zfill(4)}.jpg'
      if os.path.isfile(image_path):
        image = cv.flip(cv.imread(image_path), 1)
        result = hands.process(cv.cvtColor(image, cv.COLOR_BGR2RGB))
        keypoints = extract_keypoints(result)
        if result.multi_hand_landmarks:
          detected += 1
        np.save(supset_dir + f'/class{n_way}_sample{k_shot}/keypoints/frame_{str(frame_num).zfill(4)}.npy', keypoints)
  print(f'  Hand detected in {detected}/{num_frames} frames.')
  if detected == 0:
    print('  WARNING: No hand detected! Check lighting and make sure your hand is visible.')


def create_supset_from_keypoints(n_ways, k_shots, supset_dir):
  """
  Collects and concatenates all keypoints from a given supset_dir
  """
  num_frames = 72
  X, y = [], []
  for n in range(1, n_ways + 1):
    for s in range(1, k_shots + 1):
      sequence = []
      for frame in range(1, num_frames + 1):
        kp_path = supset_dir + f'/class{n}_sample{s}/keypoints/frame_{str(frame).zfill(4)}.npy'
        if os.path.isfile(kp_path):
          sequence.append(np.load(kp_path))
        else:
          sequence.append(np.zeros(63))
      X.append(sequence)
      y.append(n)

  X = np.array(X)
  y = np.array(y)
  for i in range(len(X)):
    X[i] = move_values(X[i], num_frames)

  support_x = torch.from_numpy(X).float()
  support_y = torch.from_numpy(y).float().type(torch.LongTensor)
  return support_x, support_y


def create_support_set(n_way, k_shot, classes):
  """
  Main function to create a support set for a given n_way, k_shot scenario
  """
  indexes = [f'#{i}' for i in range(1, K_SHOT + 1)]
  supset_state = pd.DataFrame(np.empty([K_SHOT, N_WAY], dtype=str), index=indexes, columns=GESTURES)

  supset_dir = 'support_set'
  support_x, support_y = initialize_supset_dir(n_way, k_shot, supset_dir)
  if support_x is not None:
    return support_x, support_y

  for n in range(1, n_way + 1):
    for s in range(1, k_shot + 1):
      supset_state.iloc[s - 1, n - 1] = "This turn's gesture!"
      clear_output()
      print('Current state of your support-set: \n')
      print(tabulate(supset_state, headers=supset_state.columns, tablefmt='simple', stralign='center'))
      print(f'\nPerform your gesture "{str(classes[n-1])}" into your webcam!\n')
      time.sleep(3) if n == 1 and s == 1 else time.sleep(1)

      # 비디오 파일 없이 직접 JPEG 프레임 캡처
      ok = record_frames(supset_dir + f'/class{n}_sample{s}/frames')
      if not ok:
        print('Frame capture failed.')
        return None, None

      supset_state.iloc[s - 1, n - 1] = 'v'

  clear_output()
  print('All videos have been captured. \n')
  print(tabulate(supset_state, headers=supset_state.columns, tablefmt='simple', stralign='center'))
  print('\nThe keypoints are now being extracted. This may take a few minutes.\n')

  with alive_bar(n_way * k_shot, title='Extracting keypoints...') as bar:
    for n in range(1, n_way + 1):
      for s in range(1, k_shot + 1):
        generate_supset_keypoints_from_frame(n, s, supset_dir)
        bar()

  support_x, support_y = create_supset_from_keypoints(n_way, k_shot, supset_dir)
  print('\nAll keypoints have been extracted. Your support set is ready to go!')
  return support_x, support_y


def calc_prediction(feature_encoder, relation_network, support_x, query_x, num_classes, support_num_per_class, seq_length, num_units_lstm_encoder):
  """
  Prediction function
  """
  support_features = feature_encoder(Variable(support_x).to(device))
  query_features   = feature_encoder(Variable(query_x).to(device))

  support_features = support_features.view(num_classes, support_num_per_class, seq_length, num_units_lstm_encoder)
  support_features = torch.sum(support_features, 1).squeeze(1)
  support_features_ext = support_features.unsqueeze(0).repeat(1, 1, 1, 1).to(device)
  query_features_ext   = query_features.unsqueeze(0).repeat(num_classes, 1, 1, 1)
  query_features_ext   = torch.transpose(query_features_ext, 0, 1).to(device)

  relation_pairs = torch.cat((support_features_ext, query_features_ext), 3).view(-1, seq_length, num_units_lstm_encoder * 2).to(device)
  relations = relation_network(relation_pairs).view(-1, num_classes).to(device)
  return relations


def create_prediction_data():
  """
  Creates the data necessary for a prediction (records webcam frames directly)
  """
  predict_dir = 'prediction'
  num_frames = 72

  if os.path.exists(predict_dir):
    os.system('rm -rf prediction' if os.name != 'nt' else 'rmdir /s /q prediction')

  frames_dir = predict_dir + '/frames'
  keypoints_dir = predict_dir + '/keypoints'
  os.makedirs(frames_dir)
  os.makedirs(keypoints_dir)

  print('Perform the gesture you want to detect into your webcam!')
  time.sleep(2)
  ok = record_frames(frames_dir, num_frames)
  if not ok:
    print('Frame capture failed.')
    return None

  clear_output()
  print('Frames captured. Extracting keypoints...\n')

  sequence = []
  detected = 0
  with mp.solutions.hands.Hands(
      static_image_mode=True,
      max_num_hands=1,
      min_detection_confidence=0.5) as hands:
    with alive_bar(num_frames, title='Extracting keypoints...') as bar:
      for frame_num in range(1, num_frames + 1):
        frame_path = frames_dir + f'/frame_{str(frame_num).zfill(4)}.jpg'
        if os.path.isfile(frame_path):
          image = cv.flip(cv.imread(frame_path), 1)
          result = hands.process(cv.cvtColor(image, cv.COLOR_BGR2RGB))
          keypoints = extract_keypoints(result)
          if result.multi_hand_landmarks:
            detected += 1
          np.save(keypoints_dir + f'/frame_{str(frame_num).zfill(4)}.npy', keypoints)
          sequence.append(keypoints)
        else:
          sequence.append(np.zeros(63))
        bar()

  print(f'\nHand detected in {detected}/{num_frames} frames.')
  if detected == 0:
    print('WARNING: No hand detected! Check lighting and make sure your hand is clearly visible.')

  x = np.array(sequence)
  x = move_values(x, x.shape[0])
  x = np.expand_dims(x, axis=0)
  x = torch.from_numpy(x).float()
  return x


def perform_prediction(support_x, classes, feature_encoder, relation_network):
  """
  Main function for performing a prediction
  """
  model_name = f'{len(classes)}way-{int(len(support_x) / len(classes))}shot'
  model_path = 'S-STRHanGe/deployment/models/'
  if os.path.isfile(os.path.join(model_path, f'{model_name}_deployment_param.pkl')):
    with open(os.path.join(model_path, f'{model_name}_deployment_param.pkl'), 'rb') as f:
      deployment_param = pickle.load(f)

  x = create_prediction_data()
  if x is None:
    return

  print('Predicting... \n')
  with torch.no_grad():
    relations = calc_prediction(
      feature_encoder=feature_encoder, relation_network=relation_network,
      support_x=support_x, query_x=x,
      num_classes=deployment_param['num_classes'],
      support_num_per_class=deployment_param['support_num_per_class'],
      seq_length=deployment_param['sequence_length'],
      num_units_lstm_encoder=deployment_param['num_units_lstm_encoder']
    )

  print('Prediction finished! \n')
  print(f'The model predicts: {classes[torch.argmax(relations).item()]}\n')
  print('The calculated relation scores to all classes are:\n')
  print(tabulate(relations.tolist(), tablefmt='simple', headers=classes, numalign='right', floatfmt='.2f'))
  print('\nThe higher the score, the more similarity the model sees between your gesture and the support-set.')
  print('\nRerun the cell to perform another prediction!')


def extract_all_keypoints_from_video(video_path):
  """
  Extracts keypoints from every frame of a long video (uses ffmpeg).
  """
  temp_dir = 'temp_long_video'
  frames_dir = os.path.join(temp_dir, 'frames')

  if os.path.exists(temp_dir):
    os.system(f'rm -rf {temp_dir}' if os.name != 'nt' else f'rmdir /s /q {temp_dir}')
  os.makedirs(frames_dir)

  os.system(f'ffmpeg -i "{video_path}" -qscale:v 2 -vf "scale=360:240,fps=24" {frames_dir}/frame_%05d.jpg -y')

  frame_files = sorted([f for f in os.listdir(frames_dir) if f.endswith('.jpg')])
  total_frames = len(frame_files)
  print(f'Total frames extracted: {total_frames} ({total_frames / 24:.1f} seconds)')

  all_keypoints = []
  with mp.solutions.hands.Hands(
      static_image_mode=True,
      max_num_hands=1,
      min_detection_confidence=0.5) as hands:
    with alive_bar(total_frames, title='Extracting keypoints...') as bar:
      for fname in frame_files:
        image = cv.flip(cv.imread(os.path.join(frames_dir, fname)), 1)
        result = hands.process(cv.cvtColor(image, cv.COLOR_BGR2RGB))
        all_keypoints.append(extract_keypoints(result))
        bar()

  return np.array(all_keypoints)


def temporal_nms(detections, window_size, iou_threshold=0.3):
  """
  Removes duplicate detections from overlapping windows.
  """
  if not detections:
    return []
  detections = sorted(detections, key=lambda x: x[3], reverse=True)
  kept, suppressed = [], set()
  for i, det_i in enumerate(detections):
    if i in suppressed:
      continue
    kept.append(det_i)
    for j, det_j in enumerate(detections[i + 1:], start=i + 1):
      if j in suppressed or det_i[2] != det_j[2]:
        continue
      inter = max(0, min(det_i[1], det_j[1]) - max(det_i[0], det_j[0]))
      union = window_size * 2 - inter
      if inter / union > iou_threshold:
        suppressed.add(j)
  return sorted(kept, key=lambda x: x[0])


def predict_from_long_video(video_path, support_x, classes, feature_encoder, relation_network,
                            stride=24, threshold=0.5):
  """
  Detects multiple gestures in a long video using a sliding window approach.
  """
  num_frames = 72
  fps = 24

  model_name = f'{len(classes)}way-{int(len(support_x) / len(classes))}shot'
  model_path = 'S-STRHanGe/deployment/models/'
  with open(os.path.join(model_path, f'{model_name}_deployment_param.pkl'), 'rb') as f:
    deployment_param = pickle.load(f)

  all_keypoints = extract_all_keypoints_from_video(video_path)
  total_frames = len(all_keypoints)

  if total_frames < num_frames:
    print(f'Video is too short ({total_frames} frames). At least {num_frames / fps:.1f} seconds required.')
    return []

  window_starts = list(range(0, total_frames - num_frames + 1, stride))
  detections = []

  print(f'\nRunning sliding window prediction ({len(window_starts)} windows, stride={stride} frames)...')
  with alive_bar(len(window_starts), title='Predicting...') as bar:
    for start in window_starts:
      window = all_keypoints[start:start + num_frames].copy()
      window = move_values(window, num_frames)
      x = torch.from_numpy(np.expand_dims(window, axis=0)).float()
      with torch.no_grad():
        relations = calc_prediction(
          feature_encoder=feature_encoder, relation_network=relation_network,
          support_x=support_x, query_x=x,
          num_classes=deployment_param['num_classes'],
          support_num_per_class=deployment_param['support_num_per_class'],
          seq_length=deployment_param['sequence_length'],
          num_units_lstm_encoder=deployment_param['num_units_lstm_encoder']
        )
      scores = relations[0].detach().cpu().numpy()
      best_class = int(np.argmax(scores))
      best_score = float(scores[best_class])
      if best_score >= threshold:
        detections.append((start, start + num_frames, best_class, best_score, scores.tolist()))
      bar()

  detections = temporal_nms(detections, num_frames)

  print(f'\n=== Detection Results (threshold={threshold}) ===')
  if not detections:
    print('No gestures detected. Try lowering the threshold or check your video.')
  else:
    rows = [
      [i + 1, f'{s / fps:.1f}s', f'{e / fps:.1f}s', classes[c], f'{sc:.2f}']
      for i, (s, e, c, sc, _) in enumerate(detections)
    ]
    print(tabulate(rows, headers=['#', 'Start', 'End', 'Gesture', 'Score'], tablefmt='simple'))

  return detections

print('All functions loaded successfully!')

All functions loaded successfully!


# Step 3 - 모델 설정

**N_WAY**: 분류할 제스처 클래스 수 (5 또는 10)

**K_SHOT**: 클래스당 지원 샘플 수 (1, 2, 또는 5)

**GESTURES**: 각 클래스의 이름 목록 (N_WAY 개수와 일치해야 함)

In [38]:
N_WAY = 5
K_SHOT = 1

GESTURES = ["Gesture_1",
            "Gesture_2",
            "Gesture_3",
            "Gesture_4",
            "Gesture_5"]

feature_encoder, relation_network = initialize_model(N_WAY, K_SHOT, GESTURES)

# Step 4 - Support Set 생성

모델에게 각 클래스의 샘플을 보여줍니다. 웹캠 창이 열리면 카운트다운 후 지정된 제스처를 수행하세요.

> **Windows 사용자:** `cv.imshow` 창은 Jupyter 커널이 실행 중인 동안 별도 창으로 열립니다.

In [39]:
support_set, support_labels = create_support_set(N_WAY, K_SHOT, GESTURES)

All videos have been captured. 

     Gesture_1    Gesture_2    Gesture_3    Gesture_4    Gesture_5
--  -----------  -----------  -----------  -----------  -----------
#1       v            v            v            v            v

The keypoints are now being extracted. This may take a few minutes.

on 0:   Hand detected in 45/72 frames.
on 1:   Hand detected in 64/72 frames.
on 2:   Hand detected in 63/72 frames.
on 3:   Hand detected in 68/72 frames.
on 4:   Hand detected in 47/72 frames.
Extracting keypoints... |████████████████████████████████████████| 5/5 [100%] in 23.9s (0.21/s) 

All keypoints have been extracted. Your support set is ready to go!


# Step 5 - 예측 수행

웹캠 창이 열리면 카운트다운 후 예측할 제스처를 수행하세요.

셀을 다시 실행하면 새로운 예측을 수행합니다.

In [42]:
perform_prediction(support_set, GESTURES, feature_encoder, relation_network)

Frames captured. Extracting keypoints...

Extracting keypoints... |████████████████████████████████████████| 72/72 [100%] in 2.8s (25.70/s) 

Hand detected in 47/72 frames.
Predicting... 

Prediction finished! 

The model predicts: Gesture_1

The calculated relation scores to all classes are:

  Gesture_1    Gesture_2    Gesture_3    Gesture_4    Gesture_5
-----------  -----------  -----------  -----------  -----------
       0.53         0.32         0.00         0.22         0.00

The higher the score, the more similarity the model sees between your gesture and the support-set.

Rerun the cell to perform another prediction!


# Step 6 - 제스처 탐지 및 클립 저장 (20초 녹화)

웹캠으로 20초를 녹화한 뒤, threshold 이상으로 탐지된 제스처 구간을 개별 MP4 클립으로 저장합니다.

- **CLIP_RECORD_SECONDS**: 녹화할 시간 (초)
- **CLIP_STRIDE**: 슬라이딩 윈도우 이동 간격
- **CLIP_THRESHOLD**: 탐지 임계값
- **CLIP_GAP_TOLERANCE**: 순간적 감지 실패를 메울 최대 프레임 수
- **CLIP_MIN_SEG_FRAMES**: 제스처로 인정할 최소 프레임 수
- **CLIP_OUTPUT_DIR**: 클립을 저장할 폴더

In [ ]:
CLIP_RECORD_SECONDS  = 20    # 녹화할 시간 (초), 최소 3초
CLIP_STRIDE          = 12    # 슬라이딩 윈도우 이동 간격 (프레임)
CLIP_THRESHOLD       = 0.5   # 탐지 임계값
CLIP_GAP_TOLERANCE   = 6     # 순간적 감지 실패를 메울 최대 프레임 수
CLIP_MIN_SEG_FRAMES  = 12    # 제스처로 인정할 최소 프레임 수 (~0.5초)
CLIP_OUTPUT_DIR      = 'gesture_clips'  # 클립 저장 폴더

_num_frames = 72
_fps        = 24
_frames_dir = 'temp_clip_webcam/frames'

# ── 0. 임시 디렉토리 초기화 ───────────────────────────────────────────────────
if os.path.exists('temp_clip_webcam'):
    os.system('rm -rf temp_clip_webcam' if os.name != 'nt' else 'rmdir /s /q temp_clip_webcam')
os.makedirs(_frames_dir)
os.makedirs(CLIP_OUTPUT_DIR, exist_ok=True)

# ── 1. 웹캠 녹화 ─────────────────────────────────────────────────────────────
print(f'Recording {CLIP_RECORD_SECONDS} seconds from webcam...')
print('Tip: 제스처 사이에 손을 내리면 인식률이 높아집니다!')
ok = record_frames(_frames_dir, num_frames=CLIP_RECORD_SECONDS * _fps)

if not ok:
    print('Recording failed.')
else:
    frame_files = sorted([f for f in os.listdir(_frames_dir) if f.endswith('.jpg')])
    total_frames = len(frame_files)
    print(f'\nTotal frames captured: {total_frames} ({total_frames / _fps:.1f}s)')

    # ── 2. Keypoint 추출 ──────────────────────────────────────────────────────
    all_keypoints = []
    with mp.solutions.hands.Hands(
            static_image_mode=True, max_num_hands=1,
            min_detection_confidence=0.5) as hands:
        with alive_bar(total_frames, title='Extracting keypoints...') as bar:
            for fname in frame_files:
                image = cv.flip(cv.imread(os.path.join(_frames_dir, fname)), 1)
                result = hands.process(cv.cvtColor(image, cv.COLOR_BGR2RGB))
                all_keypoints.append(extract_keypoints(result))
                bar()
    all_keypoints = np.array(all_keypoints)

    # ── 3. 손 감지 구간 탐지 ─────────────────────────────────────────────────
    has_hand = np.array([kp.max() > 0 for kp in all_keypoints])

    # 짧은 gap 메우기
    i = 0
    while i < len(has_hand):
        if not has_hand[i]:
            gap_end = i
            while gap_end < len(has_hand) and not has_hand[gap_end]:
                gap_end += 1
            if gap_end - i <= CLIP_GAP_TOLERANCE:
                has_hand[i:gap_end] = True
            i = gap_end
        else:
            i += 1

    # 연속 구간 리스트 생성
    segments = []
    in_seg = False
    for i in range(total_frames):
        if has_hand[i] and not in_seg:
            in_seg, seg_start = True, i
        elif not has_hand[i] and in_seg:
            in_seg = False
            if i - seg_start >= CLIP_MIN_SEG_FRAMES:
                segments.append((seg_start, i))
    if in_seg and total_frames - seg_start >= CLIP_MIN_SEG_FRAMES:
        segments.append((seg_start, total_frames))

    print(f'\nHand detected in {has_hand.sum()}/{total_frames} frames.')
    print(f'Found {len(segments)} gesture segment(s).')

    if not segments:
        print('\nNo segments found. Check lighting and make sure your hand is clearly visible.')
    else:
        # ── 4. 모델 파라미터 로드 ─────────────────────────────────────────────
        _model_name = f'{len(GESTURES)}way-{int(len(support_set) / len(GESTURES))}shot'
        _model_path = 'S-STRHanGe/deployment/models/'
        with open(os.path.join(_model_path, f'{_model_name}_deployment_param.pkl'), 'rb') as f:
            _dp = pickle.load(f)

        def _predict_window_clip(kp_window):
            """keypoint window → (best_class_idx, best_score, scores) or None"""
            seg_len = len(kp_window)
            padded = np.zeros((_num_frames, 63))
            padded[:min(seg_len, _num_frames)] = kp_window[:_num_frames]
            padded = move_values(padded, _num_frames)
            x = torch.from_numpy(np.expand_dims(padded, 0)).float()
            with torch.no_grad():
                relations = calc_prediction(
                    feature_encoder=feature_encoder, relation_network=relation_network,
                    support_x=support_set, query_x=x,
                    num_classes=_dp['num_classes'],
                    support_num_per_class=_dp['support_num_per_class'],
                    seq_length=_dp['sequence_length'],
                    num_units_lstm_encoder=_dp['num_units_lstm_encoder']
                )
            scores = relations[0].detach().cpu().numpy()
            best_class = int(np.argmax(scores))
            best_score = float(scores[best_class])
            if best_score >= CLIP_THRESHOLD:
                return (best_class, best_score, scores.tolist())
            return None

        # ── 5. 구간별 슬라이딩 윈도우 예측 ──────────────────────────────────
        total_windows = sum(
            1 if (e - s) < _num_frames
            else max(1, (e - s - _num_frames) // CLIP_STRIDE + 1)
            for s, e in segments
        )

        all_detections = []
        print(f'\nRunning prediction on {len(segments)} segment(s) ({total_windows} window(s))...')
        with alive_bar(total_windows, title='Predicting...') as bar:
            for seg_start, seg_end in segments:
                seg_kp  = all_keypoints[seg_start:seg_end]
                seg_len = seg_end - seg_start

                if seg_len < _num_frames:
                    result = _predict_window_clip(seg_kp)
                    if result:
                        best_class, best_score, scores = result
                        all_detections.append((seg_start, seg_end, best_class, best_score, scores))
                    bar()
                else:
                    for ws in range(0, seg_len - _num_frames + 1, CLIP_STRIDE):
                        abs_start = seg_start + ws
                        result = _predict_window_clip(seg_kp[ws:ws + _num_frames])
                        if result:
                            best_class, best_score, scores = result
                            all_detections.append((abs_start, abs_start + _num_frames, best_class, best_score, scores))
                        bar()

        all_detections = temporal_nms(all_detections, _num_frames)

        # ── 6. 결과 출력 ──────────────────────────────────────────────────────
        print(f'\n=== Detection Results (threshold={CLIP_THRESHOLD}) ===')
        if not all_detections:
            print('No gestures detected above threshold. Try lowering CLIP_THRESHOLD.')
        else:
            rows = [
                [i+1, f'{s/_fps:.1f}s', f'{e/_fps:.1f}s', GESTURES[c], f'{sc:.2f}']
                for i, (s, e, c, sc, _) in enumerate(all_detections)
            ]
            print(tabulate(rows, headers=['#', 'Start', 'End', 'Gesture', 'Score'], tablefmt='simple'))

            # ── 7. 클립 저장 ──────────────────────────────────────────────────
            fourcc = cv.VideoWriter_fourcc(*'mp4v')
            saved_clips = []

            print(f'\nSaving {len(all_detections)} clip(s) to "{CLIP_OUTPUT_DIR}/"...')
            for i, (s, e, c, sc, _) in enumerate(all_detections):
                gesture_name = GESTURES[c]
                clip_name = f'clip_{i+1:02d}_{gesture_name}_{s/_fps:.1f}s-{e/_fps:.1f}s_score{sc:.2f}.mp4'
                clip_path = os.path.join(CLIP_OUTPUT_DIR, clip_name)

                # 프레임 범위를 실제 저장된 파일 수에 맞춰 클램프
                frame_start = min(s, total_frames - 1)
                frame_end   = min(e, total_frames)

                # 첫 프레임으로 해상도 확인
                sample_frame = cv.imread(os.path.join(_frames_dir, frame_files[frame_start]))
                h, w = sample_frame.shape[:2]

                writer = cv.VideoWriter(clip_path, fourcc, float(_fps), (w, h))
                for fi in range(frame_start, frame_end):
                    frame_img = cv.imread(os.path.join(_frames_dir, frame_files[fi]))
                    if frame_img is not None:
                        writer.write(frame_img)
                writer.release()

                saved_clips.append((i+1, gesture_name, f'{sc:.2f}', clip_name))
                print(f'  Saved: {clip_name}')

            print(f'\nDone! {len(saved_clips)} clip(s) saved to "{CLIP_OUTPUT_DIR}/".')

# Step 7 - 제스처 탐지 및 클립 저장 (실시간 손 관절 시각화 포함)

웹캠으로 20초를 녹화하면서 **실시간으로 손 관절을 화면에 표시**합니다.
녹화 후 threshold 이상으로 탐지된 제스처 구간을 개별 MP4 클립으로 저장합니다.

- **CLIP_RECORD_SECONDS**: 녹화할 시간 (초)
- **CLIP_STRIDE**: 슬라이딩 윈도우 이동 간격
- **CLIP_THRESHOLD**: 탐지 임계값
- **CLIP_GAP_TOLERANCE**: 순간적 감지 실패를 메울 최대 프레임 수
- **CLIP_MIN_SEG_FRAMES**: 제스처로 인정할 최소 프레임 수
- **CLIP_OUTPUT_DIR**: 클립을 저장할 폴더

In [43]:
import threading
import queue
from collections import deque

CLIP_RECORD_SECONDS  = 20    # 녹화할 시간 (초)
CLIP_STRIDE          = 12    # 슬라이딩 윈도우 이동 간격 (프레임)
CLIP_THRESHOLD       = 0.5   # 탐지 임계값
CLIP_OUTPUT_DIR      = 'gesture_clips'  # 클립 저장 폴더

_num_frames = 72
_fps        = 24
_frames_dir = 'temp_clip_webcam/frames'

_mp_hands   = mp.solutions.hands
_mp_drawing = mp.solutions.drawing_utils
_mp_styles  = mp.solutions.drawing_styles

# ── 0. 디렉토리 초기화 ────────────────────────────────────────────────────────
if os.path.exists('temp_clip_webcam'):
    os.system('rm -rf temp_clip_webcam' if os.name != 'nt' else 'rmdir /s /q temp_clip_webcam')
os.makedirs(_frames_dir)
os.makedirs(CLIP_OUTPUT_DIR, exist_ok=True)

# ── 1. 병렬 추론을 위한 공유 상태 ────────────────────────────────────────────
_kp_queue   = queue.Queue()         # (frame_idx, keypoints) 전달
_detections = []                    # 추론 결과 누적
_det_lock   = threading.Lock()      # _detections 접근 보호
_rec_done   = threading.Event()     # 녹화 종료 신호

# ── 2. 추론 스레드 함수 ───────────────────────────────────────────────────────
def _inference_worker():
    buffer        = deque(maxlen=_num_frames)  # 최근 72프레임 keypoint 슬라이딩 버퍼
    stride_counter = 0

    _model_name = f'{len(GESTURES)}way-{int(len(support_set) / len(GESTURES))}shot'
    _model_path = 'S-STRHanGe/deployment/models/'
    with open(os.path.join(_model_path, f'{_model_name}_deployment_param.pkl'), 'rb') as f:
        dp = pickle.load(f)

    while not (_rec_done.is_set() and _kp_queue.empty()):
        try:
            frame_idx, kp = _kp_queue.get(timeout=0.05)
        except queue.Empty:
            continue

        buffer.append((frame_idx, kp))
        stride_counter += 1

        # 버퍼가 가득 차고 STRIDE 프레임마다 예측
        if len(buffer) < _num_frames or stride_counter < CLIP_STRIDE:
            continue
        stride_counter = 0

        kp_window = np.array([k for _, k in buffer])
        if kp_window.max() == 0:   # 윈도우에 손이 전혀 없으면 스킵
            continue

        kp_window = move_values(kp_window.copy(), _num_frames)
        x = torch.from_numpy(np.expand_dims(kp_window, 0)).float()
        with torch.no_grad():
            relations = calc_prediction(
                feature_encoder=feature_encoder, relation_network=relation_network,
                support_x=support_set, query_x=x,
                num_classes=dp['num_classes'],
                support_num_per_class=dp['support_num_per_class'],
                seq_length=dp['sequence_length'],
                num_units_lstm_encoder=dp['num_units_lstm_encoder']
            )
        scores     = relations[0].detach().cpu().numpy()
        best_class = int(np.argmax(scores))
        best_score = float(scores[best_class])

        if best_score >= CLIP_THRESHOLD:
            start_idx = buffer[0][0]
            end_idx   = buffer[-1][0] + 1
            with _det_lock:
                _detections.append((start_idx, end_idx, best_class, best_score, scores.tolist()))
            print(f'  [Inference] Detected: {GESTURES[best_class]} ({best_score:.2f})  '
                  f'{start_idx/_fps:.1f}s – {end_idx/_fps:.1f}s')

# ── 3. 추론 스레드 시작 ───────────────────────────────────────────────────────
inf_thread = threading.Thread(target=_inference_worker, daemon=True)
inf_thread.start()

# ── 4. 웹캠 녹화 (실시간 랜드마크 + 탐지 결과 오버레이) ──────────────────────
print(f'Recording {CLIP_RECORD_SECONDS}s  |  Inference running in background...')
print('Tip: 제스처 사이에 손을 내리면 인식률이 높아집니다!')

cap = cv.VideoCapture(0)
ok  = False

if not cap.isOpened():
    print('Error: Cannot open webcam.')
    _rec_done.set()
else:
    with _mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as hands_rt:

        # 3초 카운트다운
        cancelled = False
        for i in range(3, 0, -1):
            deadline = time.time() + 1.0
            while time.time() < deadline:
                ret, frame = cap.read()
                if not ret:
                    break
                frame = cv.flip(frame, 1)
                rgb = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
                rgb.flags.writeable = False
                results = hands_rt.process(rgb)
                rgb.flags.writeable = True
                if results.multi_hand_landmarks:
                    for lm in results.multi_hand_landmarks:
                        _mp_drawing.draw_landmarks(frame, lm, _mp_hands.HAND_CONNECTIONS,
                            _mp_styles.get_default_hand_landmarks_style(),
                            _mp_styles.get_default_hand_connections_style())
                cv.putText(frame, f'Recording starts in {i}', (30, 60),
                           cv.FONT_HERSHEY_SIMPLEX, 1.4, (0, 255, 0), 3)
                cv.imshow('Hand Gesture Recorder  [Q to cancel]', frame)
                if cv.waitKey(30) & 0xFF == ord('q'):
                    cancelled = True
                    break
            if cancelled:
                break

        # 녹화 루프
        if not cancelled:
            total_needed   = CLIP_RECORD_SECONDS * _fps
            frame_interval = 1.0 / _fps
            saved          = 0
            next_capture   = time.time()

            while saved < total_needed:
                ret, frame = cap.read()
                if not ret:
                    break
                frame = cv.flip(frame, 1)

                # 실시간 MediaPipe 처리
                rgb = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
                rgb.flags.writeable = False
                results = hands_rt.process(rgb)
                rgb.flags.writeable = True

                # 화면용 복사본에 랜드마크 그리기
                display = frame.copy()
                if results.multi_hand_landmarks:
                    for lm in results.multi_hand_landmarks:
                        _mp_drawing.draw_landmarks(display, lm, _mp_hands.HAND_CONNECTIONS,
                            _mp_styles.get_default_hand_landmarks_style(),
                            _mp_styles.get_default_hand_connections_style())

                # 24fps 기준으로 프레임 저장 + keypoint를 추론 큐에 전달
                if time.time() >= next_capture:
                    resized = cv.resize(frame, (360, 240))
                    cv.imwrite(os.path.join(_frames_dir, f'frame_{str(saved + 1).zfill(4)}.jpg'), resized)
                    kp = extract_keypoints(results)
                    _kp_queue.put((saved, kp))
                    saved        += 1
                    next_capture += frame_interval

                # 탐지 결과 오버레이 (최근 3개)
                with _det_lock:
                    recent = list(_detections[-3:])
                for ri, (s, e, c, sc, _) in enumerate(reversed(recent)):
                    label = f'{GESTURES[c]}  {sc:.2f}  ({s/_fps:.1f}s-{e/_fps:.1f}s)'
                    cv.putText(display, label, (10, display.shape[0] - 20 - ri * 28),
                               cv.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2)

                remaining = max(0, (total_needed - saved) * frame_interval)
                cv.putText(display, f'Recording ends in {remaining:.1f}s', (30, 60),
                           cv.FONT_HERSHEY_SIMPLEX, 1.4, (0, 0, 255), 3)
                cv.imshow('Hand Gesture Recorder  [Q to cancel]', display)
                if cv.waitKey(1) & 0xFF == ord('q'):
                    break

            ok = saved > 0
            print(f'\n  Captured {saved}/{total_needed} frames.')

    cap.release()
    cv.destroyAllWindows()

# ── 5. 추론 스레드 종료 대기 ──────────────────────────────────────────────────
_rec_done.set()
print('Waiting for inference thread to finish...')
inf_thread.join()
print('Done.')

# ── 6. Temporal NMS + 결과 출력 ───────────────────────────────────────────────
if ok:
    final_detections = temporal_nms(_detections, _num_frames)

    print(f'\n=== Detection Results (threshold={CLIP_THRESHOLD}) ===')
    if not final_detections:
        print('No gestures detected above threshold. Try lowering CLIP_THRESHOLD.')
    else:
        frame_files = sorted([f for f in os.listdir(_frames_dir) if f.endswith('.jpg')])
        total_frames = len(frame_files)

        rows = [
            [i+1, f'{s/_fps:.1f}s', f'{e/_fps:.1f}s', GESTURES[c], f'{sc:.2f}']
            for i, (s, e, c, sc, _) in enumerate(final_detections)
        ]
        print(tabulate(rows, headers=['#', 'Start', 'End', 'Gesture', 'Score'], tablefmt='simple'))

        # ── 7. 클립 저장 ──────────────────────────────────────────────────────
        fourcc = cv.VideoWriter_fourcc(*'mp4v')
        print(f'\nSaving {len(final_detections)} clip(s) to "{CLIP_OUTPUT_DIR}/"...')
        for i, (s, e, c, sc, _) in enumerate(final_detections):
            gesture_name = GESTURES[c]
            clip_name  = f'clip_{i+1:02d}_{gesture_name}_{s/_fps:.1f}s-{e/_fps:.1f}s_score{sc:.2f}.mp4'
            clip_path  = os.path.join(CLIP_OUTPUT_DIR, clip_name)

            frame_start = min(s, total_frames - 1)
            frame_end   = min(e, total_frames)

            sample = cv.imread(os.path.join(_frames_dir, frame_files[frame_start]))
            h, w   = sample.shape[:2]

            writer = cv.VideoWriter(clip_path, fourcc, float(_fps), (w, h))
            for fi in range(frame_start, frame_end):
                img = cv.imread(os.path.join(_frames_dir, frame_files[fi]))
                if img is not None:
                    writer.write(img)
            writer.release()
            print(f'  Saved: {clip_name}')

        print(f'\nDone! {len(final_detections)} clip(s) saved to "{CLIP_OUTPUT_DIR}/".')

Recording 20s  |  Inference running in background...
Tip: 제스처 사이에 손을 내리면 인식률이 높아집니다!
  [Inference] Detected: Gesture_4 (0.79)  2.0s – 5.0s
  [Inference] Detected: Gesture_2 (0.88)  2.5s – 5.5s
  [Inference] Detected: Gesture_2 (0.91)  3.0s – 6.0s
  [Inference] Detected: Gesture_1 (0.65)  3.5s – 6.5s
  [Inference] Detected: Gesture_1 (0.72)  4.0s – 7.0s
  [Inference] Detected: Gesture_4 (0.77)  4.5s – 7.5s
  [Inference] Detected: Gesture_4 (0.78)  5.0s – 8.0s
  [Inference] Detected: Gesture_1 (0.76)  5.5s – 8.5s
  [Inference] Detected: Gesture_1 (0.75)  6.0s – 9.0s
  [Inference] Detected: Gesture_5 (0.78)  6.5s – 9.5s
  [Inference] Detected: Gesture_5 (0.69)  7.0s – 10.0s
  [Inference] Detected: Gesture_4 (0.58)  7.5s – 10.5s
  [Inference] Detected: Gesture_5 (0.79)  8.0s – 11.0s
  [Inference] Detected: Gesture_5 (0.70)  8.5s – 11.5s
  [Inference] Detected: Gesture_3 (0.64)  9.0s – 12.0s
  [Inference] Detected: Gesture_4 (0.74)  9.5s – 12.5s
  [Inference] Detected: Gesture_4 (0.85)  10.